In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
zip_file_path = '/content/drive/MyDrive/imgs.zip'
extraction_path = '/content/state-farm-distracted-driver-detection'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

# Unzip the file using a shell command
!unzip -q {zip_file_path} -d {extraction_path}

print(f'Contents of {zip_file_path} extracted to {extraction_path}')
print('Listing extracted files:')
!ls -F {extraction_path}

Contents of /content/drive/MyDrive/imgs.zip extracted to /content/state-farm-distracted-driver-detection
Listing extracted files:
imgs/


## 1. Check GPU Availability

First, let's verify that a GPU is available, which is crucial for training deep learning models efficiently.

In [ ]:
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))

if tf.test.gpu_device_name():
    print('Default GPU Device: {}'.format(tf.test.gpu_device_name()))
else:
    print("Please install GPU version of TF")
    print("Please enable GPU runtime in Colab (Runtime -> Change runtime type -> GPU)")

TensorFlow Version: 2.20.0
Num GPUs Available: 1
Default GPU Device: /device:GPU:0


## 2. Import Libraries and Define Parameters

We'll import all necessary libraries and define key parameters for our model and dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau


# Dataset path (assuming it's in your Google Drive as specified)
DATASET_PATH = '/content/state-farm-distracted-driver-detection/imgs/train'

# Image dimensions
IMG_HEIGHT = 224
IMG_WIDTH = 224
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, 3)

# Training parameters
BATCH_SIZE = 32
EPOCHS_HEAD_TRAINING = 10  # Epochs for training only the head
EPOCHS_FINE_TUNING = 15    # Epochs for fine-tuning the whole model
VALIDATION_SPLIT = 0.2

# Output paths
SAVE_DIR = '/content/drive/MyDrive/distracted_driver_teacher_outputs'
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'best_teacher_model.keras')
CLASS_NAMES_SAVE_PATH = os.path.join(SAVE_DIR, 'class_names.json')
HISTORY_SAVE_PATH = os.path.join(SAVE_DIR, 'training_history.csv')
CM_SAVE_PATH = os.path.join(SAVE_DIR, 'confusion_matrix.png')

os.makedirs(SAVE_DIR, exist_ok=True)

## 3. Load Dataset

We'll use `tf.keras.utils.image_dataset_from_directory` to load the images directly from the class-specific subfolders. This function automatically infers labels from the directory structure.

**Please ensure that your `state-farm-distracted-driver-detection/imgs/train` directory in Google Drive contains subdirectories named `c0`, `c1`, ..., `c9`, each holding the respective images.**

In [ ]:
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split

# DATASET_PATH is already defined in a previous cell
DATA_DIR = Path(DATASET_PATH)

paths = []
labels = []

for class_dir in sorted(DATA_DIR.iterdir()):
    if class_dir.is_dir():
        for img_path in class_dir.glob("*.jpg"):
            paths.append(str(img_path))
            labels.append(class_dir.name)

df = pd.DataFrame({"path": paths, "label": labels})

train_df, val_df = train_test_split(
    df,
    test_size=VALIDATION_SPLIT,
    stratify=df["label"],
    random_state=42
)

print("Train class counts:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation class counts:")
print(val_df["label"].value_counts().sort_index())

# Define image size (already defined as IMG_HEIGHT, IMG_WIDTH)
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)

# Determine class names and map to IDs
class_names = sorted(df["label"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
NUM_CLASSES = len(class_names)
print(f"Found {NUM_CLASSES} classes: {class_names}")

# Save class names
with open(CLASS_NAMES_SAVE_PATH, 'w') as f:
    json.dump(class_names, f)
print(f"Class names saved to {CLASS_NAMES_SAVE_PATH}")

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    label = class_to_id[label.numpy().decode()]
    return image, label

def load_image_tf(path, label):
    image, label = tf.py_function(load_image, [path, label], [tf.float32, tf.int32])
    image.set_shape([IMG_HEIGHT, IMG_WIDTH, 3])
    label.set_shape([])
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["path"].values, train_df["label"].values)
).shuffle(5000).map(load_image_tf, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["path"].values, val_df["label"].values)
).map(load_image_tf, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

Train class counts:
label
c0    1991
c1    1814
c2    1853
c3    1877
c4    1861
c5    1849
c6    1860
c7    1602
c8    1529
c9    1703
Name: count, dtype: int64

Validation class counts:
label
c0    498
c1    453
c2    464
c3    469
c4    465
c5    463
c6    465
c7    400
c8    382
c9    426
Name: count, dtype: int64
Found 10 classes: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
Class names saved to /content/drive/MyDrive/distracted_driver_teacher_outputs/class_names.json


## 4. Data Augmentation

We'll apply data augmentation to increase the diversity of our training data and help the model generalize better. Horizontal flip is excluded as it might change the meaning of 'left'/'right' distractions.

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.04),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.20),
])

# Normalization layer for EfficientNetB0
preprocess_input = tf.keras.applications.efficientnet.preprocess_input

## 5. Build Teacher Model using Transfer Learning (EfficientNetB0)

We will use a pre-trained EfficientNetB0 model as our base, initialized with ImageNet weights. First, we'll train only the newly added classification head. Then, we'll fine-tune a portion of the base model.

In [ ]:
def build_teacher_model(num_classes):
    base_model = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=INPUT_SHAPE
    )

    # Freeze the base model for feature extraction
    base_model.trainable = False

    inputs = tf.keras.Input(shape=INPUT_SHAPE)
    x = augmentation(inputs) # Apply data augmentation
    x = preprocess_input(x)       # Apply EfficientNet preprocessing
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x) # Add a dropout layer for regularization
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    return model

model = build_teacher_model(NUM_CLASSES)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,062,381 (15.50 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

## 6. Callbacks

Set up callbacks for saving the best model, early stopping, and reducing the learning rate on plateau.

In [ ]:
checkpoint_callback = ModelCheckpoint(
    filepath=MODEL_SAVE_PATH,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_callback = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

callbacks = [checkpoint_callback, early_stopping_callback, reduce_lr_callback]

## 7. Train Model (Phase 1: Train Classifier Head)

First, we train only the classification head while keeping the EfficientNetB0 base frozen. This quickly adapts the new layers to our specific task.

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- Training the classification head (base frozen) ---")
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD_TRAINING,
    callbacks=callbacks
)


--- Training the classification head (base frozen) ---
Epoch 1/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step - accuracy: 0.3325 - loss: 1.9309
Epoch 1: val_accuracy improved from None to 0.70836, saving model to /content/drive/MyDrive/distracted_driver_teacher_outputs/best_teacher_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/distracted_driver_teacher_outputs/best_teacher_model.keras
561/561 ━━━━━━━━━━━━━━━━━━━━ 161s 265ms/step - accuracy: 0.4643 - loss: 1.6195 - val_accuracy: 0.7084 - val_loss: 1.1667 - learning_rate: 0.0010
Epoch 2/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.6434 - loss: 1.1545
Epoch 2: val_accuracy improved from 0.70836 to 0.75518, saving model to /content/drive/MyDrive/distracted_driver_teacher_outputs/best_teacher_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/distracted_driver_teacher_outputs/best_teacher_model.keras
561/561 ━━━━━━━━━━━━━━━━━━━━ 125s 223ms/step - accuracy: 0.6540 - loss: 1.1016 - val

## 8. Train Model (Phase 2: Fine-tuning the entire model)

Now, we'll unfreeze a portion of the base model and continue training with a lower learning rate. This allows the model to learn more task-specific features from the pre-trained layers.

In [ ]:
# Unfreeze the base model
model.trainable = True

# Unfreeze the last part of the base model as suggested
# The base_model (EfficientNetB0) is at index 2 in the model.layers list.
for layer in model.layers[2].layers[:-40]:
    layer.trainable = False

# Calculate class weights to address imbalance and boost c8/c9
# First, get class counts from the training dataframe
class_counts = train_df["label"].value_counts().sort_index()
total_samples = class_counts.sum()

# Calculate initial class weights
class_weight = {class_to_id[c_name]: total_samples / (NUM_CLASSES * count)
                for c_name, count in class_counts.items()}

# Boost weights for c8 and c9 as suggested
class_weight[class_to_id["c8"]] *= 1.3
class_weight[class_to_id["c9"]] *= 1.2

print("Calculated Class Weights:")
print(class_weight)

# It's important to recompile your model after unfreezing layers
# to make sure your changes take effect.
model.compile(
    optimizer=Adam(learning_rate=1e-5),  # Lower learning rate for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- Fine-tuning the entire model (partially unfrozen base) ---")
history_fine_tune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD_TRAINING + EPOCHS_FINE_TUNING, # Continue from previous epochs
    initial_epoch=history_head.epoch[-1] + 1,
    callbacks=callbacks,
    class_weight=class_weight # Apply class weights here
)

Calculated Class Weights:
{0: np.float64(0.9010045203415369), 1: np.float64(0.9889195148842337), 2: np.float64(0.9681057744198597), 3: np.float64(0.9557272242940863), 4: np.float64(0.9639441160666309), 5: np.float64(0.9702001081665765), 6: np.float64(0.9644623655913979), 7: np.float64(1.1197877652933832), 8: np.float64(1.525225637671681), 9: np.float64(1.2640516735173222)}

--- Fine-tuning the entire model (partially unfrozen base) ---
Epoch 11/25
561/561 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.7465 - loss: 0.8223
Epoch 11: val_accuracy improved from 0.86533 to 0.86600, saving model to /content/drive/MyDrive/distracted_driver_teacher_outputs/best_teacher_model.keras

Epoch 11: finished saving model to /content/drive/MyDrive/distracted_driver_teacher_outputs/best_teacher_model.keras
561/561 ━━━━━━━━━━━━━━━━━━━━ 156s 255ms/step - accuracy: 0.7471 - loss: 0.8115 - val_accuracy: 0.8660 - val_loss: 0.5603 - learning_rate: 1.0000e-05
Epoch 12/25
561/561 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms

## 9. Evaluate Model and Save Results

We will evaluate the trained model, plot training history, generate a confusion matrix, and print a classification report. Finally, we'll save the model and all relevant outputs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
import os

# Check if history_head and history_fine_tune are defined
# These should be defined in cells 20b40ffa and 6178fa6b respectively.
if 'history_head' in globals() and 'history_fine_tune' in globals():
    # Combine training histories
    history_df_head = pd.DataFrame(history_head.history)
    history_df_fine_tune = pd.DataFrame(history_fine_tune.history)
    history_df = pd.concat([history_df_head, history_df_fine_tune], ignore_index=True)
    history_df.to_csv(HISTORY_SAVE_PATH, index=False)
    print(f"Training history saved to {HISTORY_SAVE_PATH}")

    # Plot training history
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history_df['accuracy'], label='Training Accuracy')
    plt.plot(history_df['val_accuracy'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history_df['loss'], label='Training Loss')
    plt.plot(history_df['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

    # Evaluate the best model
    # Load the best model saved by ModelCheckpoint for final evaluation
    best_model = tf.keras.models.load_model(MODEL_SAVE_PATH, safe_mode=False)

    print("\n--- Evaluating the best teacher model ---")
    loss, accuracy = best_model.evaluate(val_ds)
    print(f"Validation Loss: {loss:.4f}")
    print(f"Validation Accuracy: {accuracy:.4f}")

    # Get predictions for confusion matrix and classification report
    y_true = np.concatenate([y for x, y in val_ds], axis=0)
    y_pred_probs = best_model.predict(val_ds)
    y_pred = np.argmax(y_pred_probs, axis=1)

    print("\n--- Confusion Matrix ---")
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.savefig(CM_SAVE_PATH)
    plt.show()
    print(f"Confusion matrix plot saved to {CM_SAVE_PATH}")

    print("\n--- Classification Report ---")
    print(classification_report(y_true, y_pred, target_names=class_names))
else:
    print("Error: Training histories (history_head, history_fine_tune) are not defined.")
    print("Please ensure that cells 20b40ffa (Phase 1 training) and 6178fa6b (Phase 2 fine-tuning) have been executed successfully before running this cell.")


Error: Training histories (history_head, history_fine_tune) are not defined.
Please ensure that cells 20b40ffa (Phase 1 training) and 6178fa6b (Phase 2 fine-tuning) have been executed successfully before running this cell.


## 10. Test Function for Single Image Prediction

This function will load an image from a given path, preprocess it, and use the trained teacher model to predict the class, confidence, and top 5 probabilities.

## 11. Explanation: Teacher-Student Distillation for ESP32

This section explains how the high-accuracy teacher model you just trained can be used to distill knowledge into a smaller, more efficient student model suitable for deployment on an ESP32 microcontroller, potentially with 96x96 grayscale or RGB INT8 models.

### What is Knowledge Distillation?

Knowledge distillation is a technique where a smaller, simpler "student" model is trained to mimic the behavior of a larger, more complex "teacher" model. The teacher model provides 'soft targets' (probability distributions over classes, which are more informative than hard labels) as supervisory signals to the student during training.

### Why for ESP32?

ESP32 microcontrollers have very limited computational power, memory, and energy. A large model like the EfficientNetB0-based teacher model is far too resource-intensive for direct deployment. Distillation allows us to create a tiny student model that retains much of the teacher's performance while being significantly smaller and faster.

### Steps for Teacher-Student Distillation for ESP32:

1.  **Define Student Model Architecture:**
    *   **Size:** Design a very small CNN architecture. For 96x96 grayscale/RGB input, it might involve a few convolutional layers with small filters, pooling, and a final dense layer. The exact architecture depends on the target performance vs. resource constraints.
    *   **Input:** The student model will be trained on 96x96 (grayscale or RGB) images, so its input layer must match this. You might need to preprocess your original 224x224 images (e.g., resize and convert to grayscale) to generate the student's training data.

2.  **Generate Soft Targets from Teacher:**
    *   Use your trained `best_teacher_model.keras` to predict probabilities (soft targets) for all images in your training dataset (and potentially a larger unlabeled dataset if available).
    *   Instead of using the hard labels (e.g., `c0`, `c1`), the student will learn from the teacher's output probability distributions (e.g., `[0.9, 0.05, 0.02, ...]`). These soft targets provide more nuanced information about class relationships and confidence.

3.  **Train the Student Model with Distillation Loss:**
    *   **Combined Loss:** The student model is trained using a loss function that combines two components:
        *   **Distillation Loss (Soft Targets):** A cross-entropy loss between the student's output probabilities and the teacher's soft targets. Often, a 'temperature' parameter is introduced to soften the teacher's probabilities further.
        *   **Student Loss (Hard Labels):** A standard cross-entropy loss between the student's output probabilities and the true (hard) labels. This ensures the student doesn't deviate too much from the ground truth.
    *   **Weighted Sum:** These two loss components are typically combined with a weighted sum, where the distillation loss often has a higher weight initially.

4.  **Quantization (INT8):**
    *   After training the student model, quantize its weights and activations to INT8 (8-bit integers). TensorFlow Lite (TFLite) supports this. Quantization drastically reduces model size and speeds up inference on hardware with INT8 capabilities (like many embedded systems including ESP32).
    *   You can perform post-training quantization or quantization-aware training.

5.  **Convert to TFLite Micro:**
    *   Convert the quantized student model to the TensorFlow Lite for Microcontrollers (TFLite Micro) format. This is specifically designed for very low-resource devices.

6.  **Deploy on ESP32:**
    *   Integrate the TFLite Micro model into your ESP32 firmware using the TFLite Micro library.
    *   The ESP32 will then run inference using its optimized kernels for the quantized model.

### Benefits:

*   **Small Footprint:** Drastically reduced model size (e.g., from tens of MBs to KBsw).
*   **Fast Inference:** Much quicker prediction times suitable for real-time applications on constrained hardware.
*   **Energy Efficiency:** Lower computational requirements lead to less power consumption.
*   **Preserved Accuracy:** Despite the size reduction, distillation helps the student model maintain a significant portion of the teacher's accuracy, making it viable for practical use.

## 12. Simple Neural Network Classifier with Adam
This new cell trains a compact CNN directly on the existing image datasets and evaluates its classification performance.


In [ ]:
# ============================================================
# SIMPLE CNN CLASSIFIER USING THE ADAM OPTIMIZER
# Run this cell after the dataset-loading cell that creates:
# train_ds, val_ds, class_names, NUM_CLASSES, IMG_HEIGHT, IMG_WIDTH
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from google.colab import files
import io
from PIL import Image

# Reproducibility
tf.keras.utils.set_random_seed(42)

# Check that the required dataset variables are available
required_variables = [
    "train_ds", "val_ds", "class_names",
    "NUM_CLASSES", "IMG_HEIGHT", "IMG_WIDTH"
]
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    raise RuntimeError(
        "Run the dataset-loading cell first. Missing variables: "
        + ", ".join(missing_variables)
    )

# Normalize pixel values from [0, 255] to [0, 1]
normalization = tf.keras.layers.Rescaling(1.0 / 255)

# A small convolutional neural network for image classification
simple_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        normalization,

        tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.20),
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="simple_driver_classifier"
)

# Adam updates the network weights during training
simple_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

simple_model.summary()

# Training controls
SIMPLE_EPOCHS = 15

simple_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

# Train the simple neural network
simple_history = simple_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=SIMPLE_EPOCHS,
    callbacks=simple_callbacks,
    verbose=1
)

# Evaluate validation performance
val_loss, val_accuracy = simple_model.evaluate(val_ds, verbose=0)
print(f"\nValidation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy * 100:.2f}%")

# Obtain predicted and true class IDs
y_true = []
y_pred = []

for images, labels in val_ds:
    probabilities = simple_model.predict(images, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    y_true.extend(labels.numpy().astype(int).tolist())
    y_pred.extend(predictions.astype(int).tolist())

y_true = np.asarray(y_true)
y_pred = np.asarray(y_pred)

# Per-class classification results
print("\nClassification Report")
print(
    classification_report(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

# Confusion matrix
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES)
)

fig, ax = plt.subplots(figsize=(10, 8))
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)
display.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
plt.title("Simple CNN - Validation Confusion Matrix")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Training curves
plt.figure(figsize=(8, 5))
plt.plot(simple_history.history["accuracy"], label="Training accuracy")
plt.plot(simple_history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Simple CNN Training")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Save the trained classifier
simple_model_path = os.path.join(
    SAVE_DIR if "SAVE_DIR" in globals() else "/content",
    "simple_cnn_adam_classifier.keras"
)
simple_model.save(simple_model_path)
print(f"\nSimple CNN model saved to: {simple_model_path}")

# Example prediction with uploaded image
print("\n--- Upload an image for prediction ---")
uploaded = files.upload()

if uploaded:
    for fn in uploaded.keys():
        print(f'User uploaded file "{fn}"')
        image_bytes = uploaded[fn]

        # Load image using PIL and convert to TensorFlow tensor
        image = Image.open(io.BytesIO(image_bytes))
        image = image.resize((IMG_HEIGHT, IMG_WIDTH)) # Resize to model's expected input size
        image_array = tf.keras.utils.img_to_array(image)
        image_array = tf.expand_dims(image_array, axis=0) # Add batch dimension

        print("\nSingle-image prediction:")
        probabilities = simple_model.predict(image_array, verbose=0)[0]
        predicted_id = int(np.argmax(probabilities))
        predicted_class_name = (
            class_names[predicted_id]
            if "class_names" in globals()
            else str(predicted_id)
        )
        confidence = float(probabilities[predicted_id])

        print(f"Predicted class: {predicted_class_name}")
        print(f"Confidence: {confidence:.4f}")
else:
    print("No image uploaded. Skipping single image prediction.")


In [ ]:
# ============================================================
# ACTIVATION FUNCTION COMPARISON
# ReLU vs Sigmoid vs Tanh vs Softmax
# Uses the already-created train_ds and val_ds datasets
# ============================================================

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google.colab import files
import io
from PIL import Image
import os

# Ensure SAVE_DIR is defined for saving models to Google Drive
# It's assumed to be defined in an earlier cell, but we'll provide a fallback.
if 'SAVE_DIR' not in globals():
    SAVE_DIR = '/content/drive/MyDrive/colab_models' # Default fallback if not defined
    os.makedirs(SAVE_DIR, exist_ok=True)

# Reproducibility
tf.keras.utils.set_random_seed(42)

# Get image shape and number of classes automatically
sample_images, sample_labels = next(iter(train_ds))
INPUT_SHAPE = tuple(sample_images.shape[1:])
NUM_CLASSES = len(class_names) if "class_names" in globals() else 10

print("Input shape:", INPUT_SHAPE)
print("Number of classes:", NUM_CLASSES)

ACTIVATIONS = ["relu", "sigmoid", "tanh", "softmax"]
EPOCHS_PER_MODEL = 10

activation_results = []
activation_histories = {}
activation_models = {}
activation_predictions = {}

def build_activation_model(hidden_activation):
    """
    Builds the same CNN architecture for every experiment.
    Only the hidden-layer activation function is changed.

    Important:
    The final output layer always uses softmax because this is
    a multi-class classification problem.
    """
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=INPUT_SHAPE),

        tf.keras.layers.Rescaling(1.0 / 255),

        tf.keras.layers.Conv2D(
            32, kernel_size=3, padding="same",
            activation=hidden_activation
        ),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(
            64, kernel_size=3, padding="same",
            activation=hidden_activation
        ),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(
            128, kernel_size=3, padding="same",
            activation=hidden_activation
        ),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.GlobalAveragePooling2D(),

        tf.keras.layers.Dense(
            128,
            activation=hidden_activation
        ),
        tf.keras.layers.Dropout(0.30),

        # Output activation is fixed as softmax for all models
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


# Store true validation/test labels once
y_true = np.concatenate(
    [labels.numpy() for _, labels in val_ds],
    axis=0
)

for activation_name in ACTIVATIONS:
    print("\n" + "=" * 75)
    print(f"TRAINING MODEL WITH HIDDEN ACTIVATION: {activation_name.upper()}")
    print("=" * 75)

    # Clear old TensorFlow graph before building the next model
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    model = build_activation_model(activation_name)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=3,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        )
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_PER_MODEL,
        callbacks=callbacks,
        verbose=1
    )

    test_loss, test_accuracy = model.evaluate(val_ds, verbose=0)

    probabilities = model.predict(val_ds, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)

    calculated_accuracy = accuracy_score(y_true, y_pred)

    activation_results.append({
        "Activation": activation_name,
        "Test Loss": test_loss,
        "Test Accuracy": test_accuracy,
        "Calculated Accuracy": calculated_accuracy,
        "Best Validation Accuracy": max(history.history["val_accuracy"]),
        "Epochs Trained": len(history.history["loss"])
    })

    activation_histories[activation_name] = history.history
    activation_models[activation_name] = model
    activation_predictions[activation_name] = y_pred

    print(f"\n{activation_name.upper()} TEST ACCURACY: {test_accuracy:.4f}")
    print(f"{activation_name.upper()} TEST LOSS: {test_loss:.4f}")

    target_names = (
        list(class_names)
        if "class_names" in globals()
        else [f"Class {i}" for i in range(NUM_CLASSES)]
    )

    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=target_names,
        zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    ))

    # Save each trained model to Google Drive
    model_save_path = os.path.join(SAVE_DIR, f"simple_cnn_{activation_name}_activation.keras")
    model.save(model_save_path)
    print(f"Model saved to: {model_save_path}")


# ------------------------------------------------------------
# RESULT COMPARISON TABLE
# ------------------------------------------------------------
results_df = pd.DataFrame(activation_results)
results_df = results_df.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 75)
print("FINAL ACTIVATION FUNCTION COMPARISON")
print("=" * 75)
display(results_df)

best_activation = results_df.loc[0, "Activation"]
best_accuracy = results_df.loc[0, "Test Accuracy"]

print(
    f"\nBest hidden activation function: {best_activation.upper()}"
    f"\nBest test accuracy: {best_accuracy:.4f}"
)


# ------------------------------------------------------------
# BAR GRAPH: TEST ACCURACY COMPARISON
# ------------------------------------------------------------
plt.figure(figsize=(9, 5))
plt.bar(
    results_df["Activation"].str.upper(),
    results_df["Test Accuracy"]
)
plt.xlabel("Hidden-Layer Activation Function")
plt.ylabel("Test / Validation Accuracy")
plt.title("Accuracy Comparison of Activation Functions")
plt.ylim(0, 1)

for index, value in enumerate(results_df["Test Accuracy"]):
    plt.text(index, value + 0.01, f"{value:.4f}", ha="center")

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# VALIDATION ACCURACY CURVES
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))

for activation_name, history_data in activation_histories.items():
    plt.plot(
        history_data["val_accuracy"],
        marker="o",
        label=activation_name.upper()
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy for Four Activation Functions")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# TRAINING ACCURACY CURVES
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))

for activation_name, history_data in activation_histories.items():
    plt.plot(
        history_data["accuracy"],
        marker="o",
        label=activation_name.upper()
    )

plt.xlabel("Epoch")
plt.ylabel("Training Accuracy")
plt.title("Training Accuracy for Four Activation Functions")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# OPTIONAL: TEST ONE IMAGE USING ALL FOUR MODELS
# ------------------------------------------------------------

print("\n--- Upload an image for prediction ---")
uploaded = files.upload()

if uploaded:
    for fn in uploaded.keys():
        print(f'User uploaded file "{fn}"')
        image_bytes = uploaded[fn]

        # Load image using PIL and convert to TensorFlow tensor
        image = Image.open(io.BytesIO(image_bytes))
        image = image.resize(INPUT_SHAPE[:2]) # Resize to model's expected input size
        image_array = tf.keras.utils.img_to_array(image)
        image_array = tf.expand_dims(image_array, axis=0) # Add batch dimension

        print("\nSingle-image predictions:")
        for activation_name, model in activation_models.items():
            probabilities = model.predict(image_array, verbose=0)[0]
            predicted_index = int(np.argmax(probabilities))
            predicted_label = (
                class_names[predicted_index]
                if "class_names" in globals()
                else str(predicted_index)
            )
            confidence = float(probabilities[predicted_index])

            print(
                f"{activation_name.upper():8s} -> "
                f"Class: {predicted_label}, "
                f"Confidence: {confidence:.4f}"
            )
else:
    print("No image uploaded. Skipping single image prediction.")
